# Setup ENCODE data
## Setup raw data from ENCODE in preparation for batch effect correction with ComBat
### Author: Martin Loza
### Date: 25/12/16

After selecting the window and genes of interest, we will setup gene pairs for co-expression analysis with ENCODE and GTEx data.

The first step is to correct batch effects and setup genes of interest. In this case we will focus only in genes with a related ENSEMBL ID

In [49]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
})

# Local variables 
seed = 777
date = "251216"

# Define colors for strand plots
red = "#E41A1C"
blue = "#090a0bff"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

encode_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENCODE/"
gene_pairs_file = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
out_dir = paste0(encode_dir,"/merged/")

# Local Functions


### Load and setup ENCODE data and metadata

In [50]:
# Get the available RNA-seq files, we will related them to the Accession numbers to transfer metadata
rna_seq_files <- list.files(file.path(encode_dir, "raw/"))
# Get the data_id from the file names
data_ids <- str_replace(rna_seq_files, pattern = ".tsv", replacement = "")

In [54]:
# Load the metadata of ENCODE data, skipping the first line
md <- read.csv2(file.path(encode_dir, "ENCODE_md_251215.tsv"), sep = "\t", header = TRUE, comment.char = "", fill = TRUE, row.names = NULL, skip = 1)

# Let's search for the RNA-seq files in the Files column and add the corresponding data_id
md$RNA_seq_file <- NA
for(i in 1:length(data_ids)){
    # Filter the metadata for the current data_id to the the corresponding Accession number
    acc <- md %>% filter(str_detect(Files, data_ids[i])) %>% select(Accession) 
    
    # Assign the data_id to the corresponding row in the metadata
    if(md$RNA_seq_file[md$Accession == acc$Accession] %>% is.na()){
        md$RNA_seq_file[md$Accession == acc$Accession] <- data_ids[i]
    } else {
        print(warning(paste0("Multiple RNA-seq files found for Accession: ", acc$Accession)))
    }

}

# Select relevant columns
sel_cols <- c('Accession','RNA_seq_file','Assay.name','Assay.title',
            'Biosample.classification','Biosample.term.name','Dbxrefs','Organism','Life.stage')
md <- md[, sel_cols]

# Let's standardize the Life.stage column
# If the Life.stage contains "adult", set it to "adult", if it contains "embryonic", set it to "embryonic", otherwise set it to "other"
md$Life.stage <- ifelse(grepl("adult", md$Life.stage, ignore.case = TRUE), "adult",
                        ifelse(grepl("embryonic", md$Life.stage, ignore.case = TRUE), "embryonic", "other"))

# Sort by Life.stage
md <- md %>%
    arrange(Life.stage)


dim(md)
head(md,2)

[1] 20  9

,Accession,RNA_seq_file,Assay.name,Assay.title,Biosample.classification,Biosample.term.name,Dbxrefs,Organism,Life.stage
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENCSR000CUE,ENCFF633DRY,RNA-seq,total RNA-seq,primary cell,articular chondrocyte of knee joint,"GEO-obsolete:GSM984611,UCSC-ENCODE-hg19:wgEncodeEH002674,GEO:GSE78607",Homo sapiens,adult
2,ENCSR000AHH,ENCFF549QYX,RNA-seq,total RNA-seq,tissue,heart,GEO:GSE87943,Homo sapiens,adult


Let's create a simple sample name

In [55]:
md$Accession

[1] "ENCSR000CUE" "ENCSR000AHH" "ENCSR000AAQ" "ENCSR000AEU" "ENCSR000CUF"
 [6] "ENCSR369RVN" "ENCSR042GYH" "ENCSR146LBD" "ENCSR000AAO" "ENCSR194HVU"
[11] "ENCSR719PXC" "ENCSR071ZLM" "ENCSR000AFL" "ENCSR000AEY" "ENCSR000AFH"
[16] "ENCSR000AFO" "ENCSR000AFI" "ENCSR000AFC" "ENCSR000AFB" "ENCSR373BDG"

In [57]:
simple_label <- c(
    "ENCSR000CUE" = "chondrocytes",
    "ENCSR000AHH" = "heart",
    "ENCSR000AAQ" = "renal_epithelial",
    "ENCSR000AEU" = "liver",
    "ENCSR000CUF" = "osteoblasts",
    "ENCSR369RVN" = "cardiac_fibroblasts",
    "ENCSR042GYH" = "ovary",
    "ENCSR146LBD" = "vagina",
    "ENCSR000AAO" = "lung_fibroblasts",
    "ENCSR194HVU" = "spleen",
    "ENCSR719PXC" = "aorta",
    "ENCSR071ZLM" = "uterus",
    "ENCSR000AFL" = "tongue",
    "ENCSR000AEY" = "frontal_cortex",
    "ENCSR000AFH" = "spinal_cord",
    "ENCSR000AFO" = "eye",
    "ENCSR000AFI" = "stomach",
    "ENCSR000AFC" = "lung",
    "ENCSR000AFB" = "liver",
    "ENCSR373BDG" = "kidney_epithelial"
)

md$simple_label <- simple_label[md$Accession]
md %>% select(simple_label, Biosample.term.name)

simple_label,Biosample.term.name
<chr>,<chr>
chondrocytes,articular chondrocyte of knee joint
heart,heart
renal_epithelial,renal cortical epithelial cell
liver,liver
osteoblasts,osteoblast
cardiac_fibroblasts,cardiac ventricle fibroblast
ovary,ovary
vagina,vagina
lung_fibroblasts,fibroblast of lung


In [58]:
# Let's create a unique sample id by merging the Biosample.term.name and Life.stage columns
md$unique_sample_id <- paste0(md$simple_label, "_", md$Life.stage)


In [59]:
cat("Any duplicated Biosample.term.name	in metadata?: ", any(duplicated(md$Biosample.term.name)), "\n")
cat("Any duplicated unique_sample_id in metadata?: ", any(duplicated(md$unique_sample_id)), "\n")

Any duplicated Biosample.term.name	in metadata?:  TRUE 
Any duplicated unique_sample_id in metadata?:  FALSE 


In [60]:
head(md)

,Accession,RNA_seq_file,Assay.name,Assay.title,Biosample.classification,Biosample.term.name,Dbxrefs,Organism,Life.stage,simple_label,unique_sample_id
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENCSR000CUE,ENCFF633DRY,RNA-seq,total RNA-seq,primary cell,articular chondrocyte of knee joint,"GEO-obsolete:GSM984611,UCSC-ENCODE-hg19:wgEncodeEH002674,GEO:GSE78607",Homo sapiens,adult,chondrocytes,chondrocytes_adult
2,ENCSR000AHH,ENCFF549QYX,RNA-seq,total RNA-seq,tissue,heart,GEO:GSE87943,Homo sapiens,adult,heart,heart_adult
3,ENCSR000AAQ,ENCFF733XOH,RNA-seq,total RNA-seq,primary cell,renal cortical epithelial cell,GEO:GSE78544,Homo sapiens,adult,renal_epithelial,renal_epithelial_adult
4,ENCSR000AEU,ENCFF960OCQ,RNA-seq,total RNA-seq,tissue,liver,GEO:GSE78562,Homo sapiens,adult,liver,liver_adult
5,ENCSR000CUF,ENCFF220VSM,RNA-seq,total RNA-seq,primary cell,osteoblast,"UCSC-ENCODE-hg19:wgEncodeEH002675,GEO-obsolete:GSM984610,GEO:GSE78608",Homo sapiens,adult,osteoblasts,osteoblasts_adult
6,ENCSR369RVN,ENCFF646YNM,RNA-seq,total RNA-seq,primary cell,cardiac ventricle fibroblast,GEO:GSE78645,Homo sapiens,adult,cardiac_fibroblasts,cardiac_fibroblasts_adult


In [65]:
# Save the md for supplementary information
out_file <- file.path("/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/", paste0("Supplementary_Table_ENCODE_datasets_", date, ".tsv"))
write.table(md, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)

Now, let's load the data, we will use the unique_sample_id for identifying each dataset

In [40]:
# Let's loop over the RNA_seq_file numbers to load the corresponding files into a list
data_list <- list()
for (acc in md$RNA_seq_file) {
    file_path <- file.path(encode_dir, paste0("raw/", acc, ".tsv"))
    data <- read.csv2(file_path, sep = "\t", header = TRUE, comment.char = "", fill = TRUE, row.names = NULL)
    # Select columns of interest: gene_id and TPM
    data <- data %>% select(gene_id, TPM)
    # Add the unique_sample_id column to the data
    unique_sample_id <- md$unique_sample_id[md$RNA_seq_file == acc]
    data$unique_sample_id <- unique_sample_id
    # Add the Accession number column to the data
    data$Accession <- md$Accession[md$RNA_seq_file == acc]
    # Add the data to the list
    data_list[[acc]] <- data
}

In [41]:
names(data_list)
head(data_list[[1]] %>% arrange(desc(TPM)) ,10)

[1] "ENCFF633DRY" "ENCFF549QYX" "ENCFF733XOH" "ENCFF960OCQ" "ENCFF220VSM"
 [6] "ENCFF646YNM" "ENCFF380WQC" "ENCFF458MLB" "ENCFF532XCI" "ENCFF612UEF"
[11] "ENCFF898UXF" "ENCFF283HWB" "ENCFF755TBK" "ENCFF931XZE" "ENCFF864MBB"
[16] "ENCFF147QBI" "ENCFF607LUW" "ENCFF678NWU" "ENCFF846SOK" "ENCFF902MQF"

,gene_id,TPM,unique_sample_id,Accession
,<chr>,<chr>,<chr>,<chr>
1,ENSG00000063046.17,99.48,articular chondrocyte of knee joint_adult,ENCSR000CUE
2,ENSG00000142173.14,99.13,articular chondrocyte of knee joint_adult,ENCSR000CUE
3,ENSG00000105640.12,98.61,articular chondrocyte of knee joint_adult,ENCSR000CUE
4,ENSG00000112096.17,98.42,articular chondrocyte of knee joint_adult,ENCSR000CUE
5,ENSG00000122406.13,97.74,articular chondrocyte of knee joint_adult,ENCSR000CUE
6,ENSG00000185624.14,97.33,articular chondrocyte of knee joint_adult,ENCSR000CUE
7,ENSG00000154380.17,96.46,articular chondrocyte of knee joint_adult,ENCSR000CUE
8,ENSG00000104904.12,96.33,articular chondrocyte of knee joint_adult,ENCSR000CUE
9,ENSG00000171858.17,96.30,articular chondrocyte of knee joint_adult,ENCSR000CUE


Let's remove the version information of the gene_id to match with ENSEMBL ids

In [42]:
# Select genes with ENSG IDs only
for (i in 1:length(data_list)) {
    data_list[[i]] <- data_list[[i]] %>%
        filter(str_detect(gene_id, pattern = "^ENSG"))
}

# Remove the version information of the ENSG IDs. This is located after the dot in the gene_id column
for (i in 1:length(data_list)) {
    data_list[[i]] <- data_list[[i]] %>%
        mutate(gene_id = str_replace(gene_id, pattern = "\\..*$", replacement = ""))
}

Let's keep only the same genes across all samples

In [43]:
# Get the list of intersecting genes across all datasets
gene_lists <- lapply(data_list, function(x) x$gene_id)
intersecting_genes <- Reduce(intersect, gene_lists)
cat("Number of intersecting genes across all datasets: ", length(intersecting_genes), "\n")

# Subset each data frame in the list to keep only the intersecting genes in the same order
for (i in 1:length(data_list)) {
    data_list[[i]] <- data_list[[i]] %>%
        filter(gene_id %in% intersecting_genes)
}

# We might have duplicated gene_ids, I believe it's because they are formed from summing up from different transcripts
# Let's remove duplicated gene_ids by keeping the one with the highest TPM
for (i in 1:length(data_list)) {
    data_list[[i]] <- data_list[[i]] %>%
        group_by(gene_id) %>%
        slice_max(order_by = TPM, n = 1, with_ties = FALSE) %>%
        ungroup() %>% 
        arrange(match(gene_id, intersecting_genes)) # match the order of intersecting_genes
}

# example in the first dataset
cat("Number of genes in the first dataset matches the number of intersecting genes: ", nrow(data_list[[1]]) == length(intersecting_genes), "\n")

Number of intersecting genes across all datasets:  58735 
Number of genes in the first dataset matches the number of intersecting genes:  TRUE 


In [44]:
cat("The order in all datasets is the same: ", all(sapply(data_list, function(x) all(x$gene_id == intersecting_genes))), "\n")

The order in all datasets is the same:  TRUE 


Let's create a log normalized list in case we need raw TPM or normalized ones

In [45]:
data_normalized <- list()

# Change the TPM to numeric
for (i in 1:length(data_list)) {
    data_list[[i]] <- data_list[[i]] %>%    
        mutate(TPM = as.numeric(TPM))
}
# log normalized the TPM values
for (i in 1:length(data_list)) {
    data_normalized[[i]] <- data_list[[i]] %>%    
        mutate(log_TPM = log1p(TPM)) 
}

Ok, now, let's merge the data

In [ ]:
# Merge the data into a single df
# Rename TPM columns with Accession numbers
for (i in 1:length(data_list)) {
    acc <- unique(data_list[[i]]$Accession)
    data_list[[i]] <- data_list[[i]] %>%
        select(gene_id, TPM) %>%
        rename(!!paste0("TPM_", acc) := TPM)

    data_normalized[[i]] <- data_normalized[[i]] %>%
        select(gene_id, log_TPM) %>%
        rename(!!paste0("log_TPM_", acc) := log_TPM)
}

# Merge all dataframes by gene_id
merged_data <- data_list[[1]]
merged_normalized <- data_normalized[[1]]
for (i in 2:length(data_list)) {
    merged_data <- left_join(merged_data, data_list[[i]], by = "gene_id")
    merged_normalized <- left_join(merged_normalized, data_normalized[[i]], by = "gene_id")
}


In [47]:
dim(merged_data)
head(merged_data, 2)
dim(merged_normalized)
head(merged_normalized, 2)

[1] 58735    21

gene_id,TPM_ENCSR000CUE,TPM_ENCSR000AHH,TPM_ENCSR000AAQ,TPM_ENCSR000AEU,TPM_ENCSR000CUF,TPM_ENCSR369RVN,TPM_ENCSR042GYH,TPM_ENCSR146LBD,TPM_ENCSR000AAO,⋯,TPM_ENCSR719PXC,TPM_ENCSR071ZLM,TPM_ENCSR000AFL,TPM_ENCSR000AEY,TPM_ENCSR000AFH,TPM_ENCSR000AFO,TPM_ENCSR000AFI,TPM_ENCSR000AFC,TPM_ENCSR000AFB,TPM_ENCSR373BDG
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000000003,4.41,2.84,16.04,5.28,1.03,5.10,5.24,0.80,1.77,⋯,1.62,0.87,2.77,0.81,5.42,10.09,3.11,19.76,13.92,10.53
ENSG00000000005,0.00,0.00,0.00,0.00,0.00,0.02,0.11,0.01,0.00,⋯,0.00,0.00,0.33,0.00,0.21,6.71,0.13,0.04,0.00,0.00


[1] 58735    21

gene_id,log_TPM_ENCSR000CUE,log_TPM_ENCSR000AHH,log_TPM_ENCSR000AAQ,log_TPM_ENCSR000AEU,log_TPM_ENCSR000CUF,log_TPM_ENCSR369RVN,log_TPM_ENCSR042GYH,log_TPM_ENCSR146LBD,log_TPM_ENCSR000AAO,⋯,log_TPM_ENCSR719PXC,log_TPM_ENCSR071ZLM,log_TPM_ENCSR000AFL,log_TPM_ENCSR000AEY,log_TPM_ENCSR000AFH,log_TPM_ENCSR000AFO,log_TPM_ENCSR000AFI,log_TPM_ENCSR000AFC,log_TPM_ENCSR000AFB,log_TPM_ENCSR373BDG
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000000003,1.688249,1.345472,2.835564,1.83737,0.7080358,1.80828877,1.83098,0.587786665,1.018847,⋯,0.9631743,0.6259384,1.3270750,0.5933268,1.8594181,2.406044,1.4134230,3.03302806,2.702703,2.444952
ENSG00000000005,0.000000,0.000000,0.000000,0.00000,0.0000000,0.01980263,0.10436,0.009950331,0.000000,⋯,0.0000000,0.0000000,0.2851789,0.0000000,0.1906204,2.042518,0.1222176,0.03922071,0.000000,0.000000


In [48]:
# Save the merged data
out_file <- file.path(out_dir, paste0("merged_raw_tpm_", date, ".tsv"))
write.table(merged_data, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)

out_file <- file.path(out_dir, paste0("merged_log_normalized_tpm_", date, ".tsv"))
write.table(merged_normalized, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)